Decisiones tomadas (modificarlas por si es necesario optimizar el modelo):
- Valor máximo extraño de número de explotaciones agrícolas: 99999.000. Podría ser considerado un outlier pero hay demasiados (189), por lo que considero que es un valor erróneo
- Si es necesario optimizar los modelos finales, se probará a tomar otras decisiones en cuanto a valores atípicos

In [3]:
import pandas as pd
import numpy as np
import pickle

import sys

sys.path.append('functions')  # Make functions/FuncionesMineria.py importable
from FuncionesMineria import (analizar_variables_categoricas, cuentaDistintos, frec_variables_num, 
                           atipicosAmissing, patron_perdidos, ImputacionCuant, ImputacionCuali)


## 1. Introducción al problema y variables implicadas
Se han elegido como variables objetivo ambas variables relacionadas con la abstención: 
- Variable objetivo continua (regresión lineal): AbstentionPtge
- Varaible objetivo binaria (regresión logística): AbstencionAlta

Por lo tanto, el objetivo del problema es crear dos modelos: 
- Una modelo de regresión lineal que permita predecir el porcentaje de abstención que hay en un municipio de España en base a datos demográficos de la región. 
- Un modelo de regresión logística que permita determinar si hay una alta abstención o no en un municipio de España. 

El resto de variables objetivo serán eliminadas

## 2. Importación del conjunto de datos 

In [101]:
datos = pd.read_excel("data/DatosEleccionesEspaña.xlsx")

datos.drop(['Izda_Pct', 'Dcha_Pct', 'Otros_Pct', 'Izquierda', 'Derecha'], 
           axis=1, inplace=True)

## 3. Análisis descriptivo del conjunto de datos. 

In [4]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8117 entries, 0 to 8116
Data columns (total 36 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name                          8117 non-null   object 
 1   CodigoProvincia               8117 non-null   int64  
 2   CCAA                          8117 non-null   object 
 3   Population                    8117 non-null   int64  
 4   TotalCensus                   8117 non-null   int64  
 5   AbstentionPtge                8117 non-null   float64
 6   AbstencionAlta                8117 non-null   int64  
 7   Age_0-4_Ptge                  8117 non-null   float64
 8   Age_under19_Ptge              8117 non-null   float64
 9   Age_19_65_pct                 8117 non-null   float64
 10  Age_over65_pct                8117 non-null   float64
 11  WomanPopulationPtge           8117 non-null   float64
 12  ForeignersPtge                8117 non-null   float64
 13  Sam

En este conjunto de datos, encontramos un total de 36 variables. A priori 32 son numéricas y 4 categóricas. 

Vamos a comprobar si hay alguna numérica que realmente sea categórica


In [5]:
cuentaDistintos(datos)

,Columna,Distintos
0,CodigoProvincia,50
1,Population,3595
2,TotalCensus,3308
3,AbstentionPtge,5673
4,AbstencionAlta,2
5,Age_0-4_Ptge,3759
6,Age_under19_Ptge,5889
7,Age_19_65_pct,6213
8,Age_over65_pct,7077
9,WomanPopulationPtge,4523


Tanto la variable explicativa CodigoProvincia como la variable objetivo AbstencionAlta son variables numéricas que realmente representan categorías. No representan un orden numérico continuo, sino una clasificación en grupos. Por ello, hay que transformarlas a categóricas

In [5]:
for var in ["CodigoProvincia","AbstencionAlta"]:
    datos[var] = datos[var].astype(str)

Ahora que ya sabemos cuales son categóricas y cuales son numéricas, las seleccionamos en variables distintas

In [8]:
# Seleccionar las columnas numéricas del DataFrame
numericas = datos.select_dtypes(include=['int', 'int32', 'int64','float', 'float32', 'float64']).columns

# Seleccionar las columnas categóricas del DataFrame
categoricas = [variable for variable in datos.columns if variable not in numericas]

Ahora, revisamos ambos tipos de variables para encontrar errores y valores extraños

Primero, las categóricas. Como se ve en el código de abajo, se encontraron los siguientes errores: 
- En la variable densidad, hay una categoría llamada "?"
- En la variable Name, esperaríamos encontrar un nombre para cada municipio y así utilizar esta variable como ID. Pero, resulta que existen dos municipios con el mismo nombre pero de comunidades diferentes. Por lo tanto, tendremos que modificar el nombre para poder utilizarlo como ID

In [9]:
analizar_variables_categoricas(datos)

{'Name':                          n         %
 Name                                
 Torrent                  2  0.000246
 Moya                     2  0.000246
 Mieres                   2  0.000246
 El Campillo              2  0.000246
 Sobrado                  2  0.000246
 ...                     ..       ...
 Vizcaínos                1  0.000123
 Zael                     1  0.000123
 Zarzosa de Río Pisuerga  1  0.000123
 Zazuar                   1  0.000123
 Almoharín                1  0.000123
 
 [8100 rows x 2 columns],
 'CCAA':                    n         %
 CCAA                          
 CastillaLeón    2248  0.276950
 Cataluña         947  0.116669
 CastillaMancha   919  0.113219
 Andalucía        773  0.095232
 Aragón           731  0.090058
 ComValenciana    542  0.066773
 Extremadura      387  0.047678
 Galicia          314  0.038684
 Navarra          272  0.033510
 PaísVasco        251  0.030923
 Madrid           179  0.022052
 Rioja            174  0.021436
 Cantabria    

Ahora, analizamos las variables numéricas: 
- Age_19_65_pct: su valor máximo es 100.002, investigar
- Age_over65_pct: tiene un valor mínimo negativo (-18.052).  
- ForeignersPtge: lo mismo, un valor mínimo negativo (-8.960)

Observación: No podemos transformar todos los valores negativos de porcentajes en np.nan ya que la variable PobChange_pct tiene porcentajes negativos que indican la disminución de población respecto a las anteriores elecciones

- SameComAutonPtge: valor máximo de 127.156. No puede ser ya que es un porcentaje
- Explotaciones: Valor máximo extraño de número de explotaciones agrícolas: 99999.000.


In [10]:
descripcion_num = datos.describe()
for num in numericas:
    descripcion_num.loc["Asimetria", num] = datos[num].skew()
    descripcion_num.loc["Kurtosis",num ] = datos[num].kurtosis()
    descripcion_num.loc["Rango", num] = datos[num].max() - datos[num].min()

# Configurar opciones de visualización de pandas
descripcion_num = descripcion_num.round(3)

descripcion_num.T

,count,mean,std,min,25%,50%,75%,max,Asimetria,Kurtosis,Rango
CodigoProvincia,8117.0,26.665,14.893,1.000,13.000,26.000,41.000,50.000,0.010,-1.323,49.000
Population,8117.0,5722.345,46204.176,5.000,166.000,548.000,2427.000,3141991.000,46.041,2820.327,3141986.000
TotalCensus,8117.0,4247.864,34423.439,5.000,140.000,447.000,1843.000,2363829.000,46.545,2893.453,2363824.000
AbstentionPtge,8117.0,26.502,7.533,0.000,21.678,26.424,31.471,57.576,-0.054,0.494,57.576
Age_0-4_Ptge,8117.0,3.018,2.053,0.000,1.389,2.975,4.533,13.245,0.344,-0.207,13.245
Age_under19_Ptge,8117.0,13.564,6.777,0.000,8.334,13.881,19.055,33.696,-0.105,-0.792,33.696
Age_19_65_pct,8117.0,57.371,6.819,23.459,53.845,58.655,61.818,100.002,-0.814,2.156,76.543
Age_over65_pct,8117.0,29.065,11.767,-18.052,19.827,27.559,36.911,76.472,0.585,0.102,94.524
WomanPopulationPtge,8117.0,47.302,4.362,11.765,45.725,48.485,50.000,72.683,-1.671,5.801,60.918
ForeignersPtge,8117.0,5.618,7.349,-8.960,1.060,3.590,8.180,71.470,2.498,11.357,80.430


## 4. Corrección de errores detectados

Para las variables categóricas, teníamos: 
- En la variable densidad, hay una categoría llamada "?"
- En la variable Name, esperaríamos encontrar un nombre para cada municipio y así utilizar esta variable como ID. Pero, resulta que existen dos municipios con el mismo nombre pero de comunidades diferentes. Por lo tanto, tendremos que modificar el nombre para poder utilizarlo como ID

In [11]:
datos['Densidad'] = datos['Densidad'].replace('?', np.nan)

# Identificar los nombres duplicados
nombres_duplicados = datos[datos.duplicated(subset=['Name'], keep=False)]

for nombre in nombres_duplicados['Name'].unique():
    mascara = datos['Name'] == nombre
    datos.loc[mascara, 'Name'] = datos.loc[mascara, 'Name'] + '_' + datos.loc[mascara, 'CCAA']


In [12]:
# Verificar los cambios
print("Nombres que fueron modificados (ordenados alfabéticamente):")
print(datos[datos['Name'].str.contains('_')][['Name', 'CCAA']].sort_values('Name').to_string())

Nombres que fueron modificados (ordenados alfabéticamente):
                                           Name            CCAA
21                    Arroyomolinos_Extremadura     Extremadura
2749                       Arroyomolinos_Madrid          Madrid
935                            Cabanes_Cataluña        Cataluña
296                       Cabanes_ComValenciana   ComValenciana
729                     Castejón_CastillaMancha  CastillaMancha
3136                           Castejón_Navarra         Navarra
4227                            Cieza_Cantabria       Cantabria
3035                               Cieza_Murcia          Murcia
1704                      El Campillo_Andalucía       Andalucía
5754                   El Campillo_CastillaLeón    CastillaLeón
4854                          El Molar_Cataluña        Cataluña
2789                            El Molar_Madrid          Madrid
5075                             Fonfría_Aragón          Aragón
6100                       Fonfría_CastillaL

In [13]:
#Verificar que la variable Name tiene 8117 valores distintos, demostrando así que son únicos
#y que la variable puede transformarse en ID
print("Número de valores únicos en Name:", datos['Name'].nunique())
print("Contador de valores de la variable densidad:", datos["Densidad"].value_counts())

Número de valores únicos en Name: 8117
Contador de valores de la variable densidad: Densidad
MuyBaja    6416
Baja       1053
Alta        556
Name: count, dtype: int64


Ahora sí, establecemos Name como ID

In [14]:

datos = datos.set_index('Name')


Para las variables numéricas, tenemos: 
- Age_19_65_pct: su valor máximo es 100.002, investigar
- Age_over65_pct: tiene un valor mínimo negativo (-18.052).  
- ForeignersPtge: lo mismo, un valor mínimo negativo (-8.960)
- SameComAutonPtge: valor máximo de 127.156. No puede ser ya que es un porcentaje

Observación: No podemos transformar todos los valores negativos de porcentajes en np.nan ya que la variable PobChange_pct tiene porcentajes negativos que indican la disminución de población respecto a las anteriores elecciones


- Explotaciones: Valor máximo extraño de número de explotaciones agrícolas: 99999.000. Podría ser considerado un outlier pero hay demasiados (189), por lo que considero que es un valor erróneo

In [15]:
# Lista de variables a corregir
variables_porcentaje = ['Age_19_65_pct', 'Age_over65_pct', 'ForeignersPtge', "SameComAutonPtge"]

# Transformar valores negativos en np.nan y valores > 100 en np.nan
for var in variables_porcentaje:
    # Valores negativos a np.nan
    datos.loc[datos[var] < 0, var] = np.nan
    # Valores mayores a 100 a np.nan (son porcentajes)
    datos.loc[datos[var] > 100, var] = np.nan


datos['Explotaciones'] = datos['Explotaciones'].replace(99999, np.nan)



In [16]:
datos.describe().T

,count,mean,std,min,25%,50%,75%,max
CodigoProvincia,8117.0,26.664654,14.893449,1.0000,13.00000,26.0000,41.00000,50.000
Population,8117.0,5722.344709,46204.175926,5.0000,166.00000,548.0000,2427.00000,3141991.000
TotalCensus,8117.0,4247.863989,34423.439346,5.0000,140.00000,447.0000,1843.00000,2363829.000
AbstentionPtge,8117.0,26.501647,7.533438,0.0000,21.67800,26.4240,31.47100,57.576
Age_0-4_Ptge,8117.0,3.018274,2.052625,0.0000,1.38900,2.9750,4.53300,13.245
Age_under19_Ptge,8117.0,13.564087,6.777445,0.0000,8.33400,13.8810,19.05500,33.696
Age_19_65_pct,8116.0,57.365339,6.802614,23.4590,53.84500,58.6545,61.81800,100.000
Age_over65_pct,8114.0,29.078978,11.746858,0.0000,19.83225,27.5635,36.91475,76.472
WomanPopulationPtge,8117.0,47.302297,4.362347,11.7650,45.72500,48.4850,50.00000,72.683
ForeignersPtge,7464.0,6.342925,7.194970,0.0000,1.62000,4.1300,8.81000,71.470


Seleccionamos variables objetivo y de entrada, además de crear listas para las variables de entrada

In [17]:
varObjCont = datos['AbstentionPtge'] #variable continua que predeciremos con una regresión lineal
varObjBin = datos['AbstencionAlta'] #variable binaria que predeciremos con una regresión logística
datos_input = datos.drop(['AbstentionPtge', 'AbstencionAlta'], axis = 1)


# Genera una lista con los nombres de las variables del cojunto de datos input.
variables_input = list(datos_input.columns)  

# Selecionamos las variables numéricas
numericas_input = datos_input.select_dtypes(include = ['int', 'int32', 'int64','float', 'float32', 'float64']).columns

# Selecionamos las variables categóricas
categoricas_input = [variable for variable in variables_input if variable not in numericas_input]

## 5. Análisis de valores atípicos. Decisiones.


En primer lugar, analizamos el porcentaje de valores atípicos por variable numérica.

In [18]:
resultados = {x: atipicosAmissing(datos_input[x])[1] / len(datos_input) for x in numericas_input}
resultados 

{'CodigoProvincia': 0.0,
 'Population': 0.0990513736602193,
 'TotalCensus': 0.09634101268941728,
 'Age_0-4_Ptge': 0.0,
 'Age_under19_Ptge': 0.0,
 'Age_19_65_pct': 0.002833559196747567,
 'Age_over65_pct': 0.0,
 'WomanPopulationPtge': 0.002587162744856474,
 'ForeignersPtge': 0.0,
 'SameComAutonPtge': 0.0,
 'SameComAutonDiffProvPtge': 0.020327707281015153,
 'DifComAutonPtge': 0.004927929037821855,
 'UnemployLess25_Ptge': 0.003203153874584206,
 'Unemploy25_40_Ptge': 0.0,
 'UnemployMore40_Ptge': 0.0,
 'AgricultureUnemploymentPtge': 0.019958112603178514,
 'IndustryUnemploymentPtge': 0.005913514845386226,
 'ConstructionUnemploymentPtge': 0.006529505975113958,
 'ServicesUnemploymentPtge': 0.0,
 'totalEmpresas': 0.0,
 'Industria': 0.0,
 'Construccion': 0.0,
 'ComercTTEHosteleria': 0.0,
 'Servicios': 0.0,
 'inmuebles': 0.0,
 'Pob2010': 0.0,
 'SUPERFICIE': 0.0,
 'PobChange_pct': 0.0,
 'PersonasInmueble': 0.0,
 'Explotaciones': 0.0}

Debido a estos porcentajes y al significado de cada variable, se decidió lo siguiente: 
- Mantener los valores atípicos de población y de censo ya que es normal que haya poblaciones con mayor o menor tamaño, sobre todo si comparamos un municipio pequeño con ciudades como Valencia o Barcelona
- Los demás valores atípicos pertenecen a porcentajes. Por ahora, se ha decidido sustituirlos por valores missing. 

Si es necesario optimizar los modelos finales, se probará a tomar otras decisiones en cuanto a valores atípicos

In [19]:
# Modifico los atipicos como missings. 

variables_a_modificar = [var for var in numericas_input if var not in ['Population', 'TotalCensus']]

for variable_numérica in variables_a_modificar:
    datos_input[variable_numérica] = atipicosAmissing(datos_input[variable_numérica])[0]

## 6. Análisis de valores perdidos. Imputaciones.

A nivel de columna, analizamos la cantidad de NaNs y su proporción

In [20]:
prop_missingsVars = datos_input.isna().sum()/len(datos_input)
prop_missingsVars

CodigoProvincia                 0.000000
CCAA                            0.000000
Population                      0.000000
TotalCensus                     0.000000
Age_0-4_Ptge                    0.000000
Age_under19_Ptge                0.000000
Age_19_65_pct                   0.002957
Age_over65_pct                  0.000370
WomanPopulationPtge             0.002587
ForeignersPtge                  0.080448
SameComAutonPtge                0.000370
SameComAutonDiffProvPtge        0.020328
DifComAutonPtge                 0.004928
UnemployLess25_Ptge             0.003203
Unemploy25_40_Ptge              0.000000
UnemployMore40_Ptge             0.000000
AgricultureUnemploymentPtge     0.019958
IndustryUnemploymentPtge        0.005914
ConstructionUnemploymentPtge    0.006530
ServicesUnemploymentPtge        0.000000
totalEmpresas                   0.000616
Industria                       0.023161
Construccion                    0.017125
ComercTTEHosteleria             0.001109
Servicios       

No hay ninguna variable con más de la mitad de datos perdidos, por lo que no eliminamos ninguna.

A nivel de fila:



In [21]:
# Creamos la variable prop_missings que recoge la proporción de valores perdidos por cada observación
datos_input['prop_missings'] = datos_input.isna().mean(axis=1)

# Realizamos un estudio descriptivo básico de la nueva variable
print(datos_input['prop_missings'].describe())

print(len(datos_input['prop_missings'].unique()))


count    8117.000000
mean        0.008139
std         0.019482
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.333333
Name: prop_missings, dtype: float64
8


Sin embargo, no eliminamos ninguna observación porque el total de missings de cada observación nunca supera el 50% 

In [22]:
eliminar = datos_input['prop_missings'] > 0.5
eliminar.value_counts()

prop_missings
False    8117
Name: count, dtype: int64

Solo hay 8 valores distintos en prop_missings. La transformamos en categórica. De esta forma, recogemos información de la proporción de valores perdidos que hay por observación, clasificando cada una en 8 grupos diferentes.

In [23]:

datos_input["prop_missings"] = datos_input["prop_missings"].astype(str)

variables_input.append('prop_missings')
categoricas_input.append('prop_missings')

Imputaciones de valores:

En este conjunto de datos, contamos con varios grupos de variables que son complementarias. Es decir, conocemos el valor de la suma de los valores de ciertas variables. Por ejemplo, sabemos que los grupos de variables que representan porcentajes (como la edad) deben sumar 100. Lo mismo ocurre con las variables relacionadas con el numero de empresas, el total debe sumar lo que indica la variable totalEmpresas.

Si solo falta un valor en ese grupo de variables (por observación), se puede calcular restándolo de 100. En el caso de los grupos estipulados como grupo1, grupo2 y grupo3, solo hay una fila con más de un valor faltante. El resto, pueden imputarse restando el total a 100 de forma sencilla

In [24]:
# Definición de grupos
grupo1 = ['Age_under19_Ptge', 'Age_19_65_pct', 'Age_over65_pct']
grupo2 = ['SameComAutonPtge', 'SameComAutonDiffProvPtge', 'DifComAutonPtge']
grupo3 = ['UnemployLess25_Ptge', 'Unemploy25_40_Ptge', 'UnemployMore40_Ptge']

# Función para imprimir filas con más de un NaN en un grupo
def check_missing_in_group(df, group, group_name):

    missing_count = df[group].isnull().sum(axis=1)
    # Selecciona las filas que tienen más de un NaN
    filas_con_muchos_nan = df[missing_count > 1]
    if not filas_con_muchos_nan.empty:
        print(f"En {group_name} se encontraron {len(filas_con_muchos_nan)} filas con más de un valor faltante:")
        print(filas_con_muchos_nan[group])
    else:
        print(f"No se encontraron filas con más de un valor faltante en {group_name}.")

# Aplicar la función a cada grupo
check_missing_in_group(datos, grupo1, "Porcentajes de edad")
check_missing_in_group(datos, grupo2, "Porcentajes de residencia")
check_missing_in_group(datos, grupo3, "Porcentajes de desempleo por edad")


En Porcentajes de edad se encontraron 1 filas con más de un valor faltante:
                Age_under19_Ptge  Age_19_65_pct  Age_over65_pct
Name                                                           
Illán de Vacas               0.0            NaN             NaN
No se encontraron filas con más de un valor faltante en Porcentajes de residencia.
No se encontraron filas con más de un valor faltante en Porcentajes de desempleo por edad.


Imputación de valores complementarios: 

In [25]:

# Función para rellenar el valor faltante en un grupo de variables complementarias
def fill_complementary_group(row, columns):


    missing_count = row[columns].isnull().sum()
    if missing_count == 1:

        sum_present = row[columns].sum(skipna=True)
        missing_value = 100 - sum_present
        missing_column = row[columns].isnull().idxmax()
        row[missing_column] = missing_value

    return row

# Definición de los grupos complementarios:

# Grupo 1: Porcentajes de edad
grupo1 = ['Age_under19_Ptge', 'Age_19_65_pct', 'Age_over65_pct']

# Grupo 2: Porcentajes de residencia
grupo2 = ['SameComAutonPtge', 'SameComAutonDiffProvPtge', 'DifComAutonPtge']

# Grupo 3: Porcentajes de desempleo por edad
grupo3 = ['UnemployLess25_Ptge', 'Unemploy25_40_Ptge', 'UnemployMore40_Ptge']

# Aplicamos la función a cada fila del DataFrame para cada grupo
for grupo in [grupo1, grupo2, grupo3]:
    datos = datos.apply(lambda row: fill_complementary_group(row, grupo), axis=1)


In [26]:
#Mostramos el porcentaje de valores faltantes por columna luego de la imputación
porcentaje_faltantes = datos.isnull().mean() * 100
print("Porcentaje de valores faltantes por columna:")
print(porcentaje_faltantes)


Porcentaje de valores faltantes por columna:
CodigoProvincia                 0.000000
CCAA                            0.000000
Population                      0.000000
TotalCensus                     0.000000
AbstentionPtge                  0.000000
AbstencionAlta                  0.000000
Age_0-4_Ptge                    0.000000
Age_under19_Ptge                0.000000
Age_19_65_pct                   0.012320
Age_over65_pct                  0.012320
WomanPopulationPtge             0.000000
ForeignersPtge                  8.044844
SameComAutonPtge                0.000000
SameComAutonDiffProvPtge        0.000000
DifComAutonPtge                 0.000000
UnemployLess25_Ptge             0.000000
Unemploy25_40_Ptge              0.000000
UnemployMore40_Ptge             0.000000
AgricultureUnemploymentPtge     0.000000
IndustryUnemploymentPtge        0.000000
ConstructionUnemploymentPtge    0.000000
ServicesUnemploymentPtge        0.000000
totalEmpresas                   0.061599
Industria   

Sin embargo, en el caso de los grupos 4 y 5, parece que falta una columna para cubrir todos los
porcenajes (grupo4) o el total de empresas (grupo5). Por ejemplo, para el grupo4, en muchas observaciones el total no suma 100, ya que parece que falta otra columna que representa el resto de porcentaje de desempleados en otros sectores. 


In [27]:

grupo4 = ['AgricultureUnemploymentPtge', 'IndustryUnemploymentPtge',
                       'ConstructionUnemploymentPtge', 'ServicesUnemploymentPtge']


grupo5 = [ 'totalEmpresas', 'Industria', 'Construccion', 'ComercTTEHosteleria', 
          'Servicios']

datos[grupo5].sample(5)

,totalEmpresas,Industria,Construccion,ComercTTEHosteleria,Servicios
Name,,,,,
Roda de Eresma,17.0,0.0,0.0,0.0,0.0
Mahora,80.0,11.0,12.0,40.0,17.0
Cascante del Río,0.0,0.0,0.0,0.0,0.0
Cazalegas,141.0,9.0,26.0,69.0,37.0
Hinojos,140.0,12.0,12.0,78.0,38.0



La lógica nos dice que "existe" una columna faltante que representa esos datos complementarios. Por lo tanto, si no hay más NaN en una observación, podríamos calcular esa columna. Sin embargo, como no podemos confirmar con el conjunto de datos que esa columna sea real, en este caso se procede con las imputaciones vistas en clase. 

In [29]:
## IMPUTACIONES
# Imputo todas las cuantitativas, seleccionar el tipo de imputacion: media, mediana o aleatorio
for x in numericas_input:
    datos_input[x] = ImputacionCuant(datos_input[x], 'aleatorio')

# Imputo todas las cualitativas, seleccionar el tipo de imputacion: moda o aleatorio
for x in categoricas_input:
    datos_input[x] = ImputacionCuali(datos_input[x], 'aleatorio')

# Reviso que no queden datos missings
datos_input.isna().sum()

CodigoProvincia                 0
CCAA                            0
Population                      0
TotalCensus                     0
Age_0-4_Ptge                    0
Age_under19_Ptge                0
Age_19_65_pct                   0
Age_over65_pct                  0
WomanPopulationPtge             0
ForeignersPtge                  0
SameComAutonPtge                0
SameComAutonDiffProvPtge        0
DifComAutonPtge                 0
UnemployLess25_Ptge             0
Unemploy25_40_Ptge              0
UnemployMore40_Ptge             0
AgricultureUnemploymentPtge     0
IndustryUnemploymentPtge        0
ConstructionUnemploymentPtge    0
ServicesUnemploymentPtge        0
totalEmpresas                   0
Industria                       0
Construccion                    0
ComercTTEHosteleria             0
Servicios                       0
ActividadPpal                   0
inmuebles                       0
Pob2010                         0
SUPERFICIE                      0
Densidad      

In [30]:

datosEleccionesLimpios = pd.concat([varObjBin, varObjCont, datos_input], axis = 1)
with open('data_clean/datosEleccionesDep.pickle', 'wb') as archivo:
    pickle.dump(datosEleccionesLimpios, archivo)